In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python

import numpy as np
import pandas as pd
import os, glob, math, random, time

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# -----------------------
# 0) Repro + device
# -----------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

# -----------------------
# 1) Find competition data paths automatically
# -----------------------
def find_comp_paths(root="/kaggle/input"):
    raw_candidates = []
    test_candidates = []
    for r, d, f in os.walk(root):
        base = os.path.basename(r)
        if base == "raw" and "lat_long.npy" in f:
            raw_candidates.append(r)
        if base == "test_in" and "cpm25.npy" in f:
            test_candidates.append(r)
    raw_candidates = sorted(raw_candidates, key=len)
    test_candidates = sorted(test_candidates, key=len)
    raw_path = raw_candidates[0] if raw_candidates else None
    test_path = test_candidates[0] if test_candidates else None
    return raw_path, test_path

RAW_PATH, TEST_PATH = find_comp_paths()
print("RAW_PATH:", RAW_PATH)
print("TEST_PATH:", TEST_PATH)
if RAW_PATH is None or TEST_PATH is None:
    raise RuntimeError(
        "Could not find competition dataset. In the right panel, click Add data and attach the competition dataset."
    )

MONTHS_ALL = sorted([m for m in os.listdir(RAW_PATH) if os.path.isdir(os.path.join(RAW_PATH, m))])
print("Month folders in raw:", MONTHS_ALL)

MONTHS = [m for m in ['APRIL_16', 'JULY_16', 'OCT_16', 'DEC_16'] if m in MONTHS_ALL]
if len(MONTHS) == 0:
    MONTHS = [m for m in MONTHS_ALL if m.endswith("_16")][:4]
print("Using months:", MONTHS)

LATLON_PATH = os.path.join(RAW_PATH, "lat_long.npy")
latlon = np.load(LATLON_PATH, mmap_mode="r")
print("lat_long.npy shape:", latlon.shape)

def prepare_latlon(latlon_arr):
    x = np.array(latlon_arr, dtype=np.float32)
    if x.ndim == 3 and x.shape[-1] == 2:
        lat = x[..., 0]
        lon = x[..., 1]
    elif x.ndim == 3 and x.shape[0] == 2:
        lat = x[0]
        lon = x[1]
    else:
        raise ValueError(f"Unexpected latlon shape: {x.shape}")
    lat = 2.0 * (lat - lat.min()) / (lat.max() - lat.min() + 1e-6) - 1.0
    lon = 2.0 * (lon - lon.min()) / (lon.max() - lon.min() + 1e-6) - 1.0
    return lat.astype(np.float32), lon.astype(np.float32)

LAT_CH, LON_CH = prepare_latlon(latlon)
H, W = LAT_CH.shape
print("Grid:", H, W)

# -----------------------
# 2) Feature lists
# -----------------------
MET = ['q2','t2','u10','v10','swdown','pblh','psfc','rain']
EMS = ['PM25','NH3','SO2','NOx','NMVOC_e','NMVOC_finn','bio']
COVARS = MET + EMS
TARGET = "cpm25"

# -----------------------
# 3) Efficient month-wise memmaps + time-aware split
# -----------------------
T_TOTAL = 26
T_CTX   = 10
T_FCST  = 16
assert T_CTX + T_FCST == T_TOTAL

def load_month_memmaps(month):
    base = os.path.join(RAW_PATH, month)
    mm = {}
    for feat in [TARGET] + COVARS:
        path = os.path.join(base, f"{feat}.npy")
        if not os.path.exists(path):
            raise FileNotFoundError(f"Missing {path}")
        mm[feat] = np.load(path, mmap_mode="r")
    time_path = os.path.join(base, "time.npy")
    if os.path.exists(time_path):
        mm["time"] = np.load(time_path, mmap_mode="r")
    return mm

MONTH_DATA = {m: load_month_memmaps(m) for m in MONTHS}
for m in MONTHS:
    print(m, TARGET, MONTH_DATA[m][TARGET].shape)

def build_window_indices(months, val_frac=0.2):
    train_idx = []
    val_idx = []
    for m in months:
        Tm = MONTH_DATA[m][TARGET].shape[0]
        max_start = Tm - T_TOTAL
        if max_start < 1:
            continue
        cutoff_t = int(Tm * (1.0 - val_frac))
        split_start = max(0, cutoff_t - T_TOTAL)
        for t in range(0, max_start + 1):
            if t >= split_start:
                val_idx.append((m, t))
            else:
                train_idx.append((m, t))
    return train_idx, val_idx

def build_all_window_indices(months):
    all_idx = []
    for m in months:
        Tm = MONTH_DATA[m][TARGET].shape[0]
        max_start = Tm - T_TOTAL
        if max_start < 1:
            continue
        for t in range(0, max_start + 1):
            all_idx.append((m, t))
    return all_idx

TRAIN_IDX, VAL_IDX = build_window_indices(MONTHS, val_frac=0.2)
FULL_IDX = build_all_window_indices(MONTHS)

print("Train windows:", len(TRAIN_IDX), "Val windows:", len(VAL_IDX), "Full windows:", len(FULL_IDX))

# -----------------------
# 4) Compute normalization from TRAIN ONLY
# -----------------------
def compute_mean_std_from_indices(indices, sample_count=500, pix_per_sample=2048):
    feats = [TARGET] + COVARS
    sums = {f: 0.0 for f in feats}
    sqs  = {f: 0.0 for f in feats}
    n    = {f: 0   for f in feats}

    rng = np.random.default_rng(SEED)

    for _ in range(sample_count):
        m, t0 = indices[rng.integers(0, len(indices))]
        tt = int(rng.integers(0, T_TOTAL))
        ys = rng.integers(0, H, size=pix_per_sample)
        xs = rng.integers(0, W, size=pix_per_sample)
        for f in feats:
            arr = MONTH_DATA[m][f][t0 + tt]
            vals = arr[ys, xs].astype(np.float64)
            sums[f] += float(vals.sum())
            sqs[f]  += float((vals * vals).sum())
            n[f]    += int(vals.size)

    mean = {}
    std = {}
    for f in feats:
        mu = sums[f] / max(1, n[f])
        var = sqs[f] / max(1, n[f]) - mu * mu
        var = max(var, 1e-8)
        mean[f] = float(mu)
        std[f] = float(math.sqrt(var))
    return mean, std

print("Computing train-only normalization stats (fast sampling)...")
MEAN, STD = compute_mean_std_from_indices(TRAIN_IDX, sample_count=500, pix_per_sample=2048)
print("Example stats:", {k: (round(MEAN[k], 3), round(STD[k], 3)) for k in list(MEAN.keys())[:5]})

def norm(x, feat):
    return (x - MEAN[feat]) / (STD[feat] + 1e-6)

def denorm(x, feat):
    return x * (STD[feat] + 1e-6) + MEAN[feat]

def compute_engineered_stats_from_indices(indices, sample_count=500, pix_per_sample=2048):
    rng = np.random.default_rng(SEED)
    ws_sum = 0.0
    ws_sqs = 0.0
    vt_sum = 0.0
    vt_sqs = 0.0
    n = 0

    for _ in range(sample_count):
        m, t0 = indices[rng.integers(0, len(indices))]
        tt = int(rng.integers(0, T_TOTAL))
        ys = rng.integers(0, H, size=pix_per_sample)
        xs = rng.integers(0, W, size=pix_per_sample)

        u = MONTH_DATA[m]["u10"][t0 + tt][ys, xs].astype(np.float64)
        v = MONTH_DATA[m]["v10"][t0 + tt][ys, xs].astype(np.float64)
        p = MONTH_DATA[m]["pblh"][t0 + tt][ys, xs].astype(np.float64)

        ws = np.sqrt(u*u + v*v + 1e-6)
        vt = ws * p

        ws_sum += float(ws.sum())
        ws_sqs += float((ws * ws).sum())
        vt_sum += float(vt.sum())
        vt_sqs += float((vt * vt).sum())
        n += int(ws.size)

    ws_mean = ws_sum / max(1, n)
    ws_var = ws_sqs / max(1, n) - ws_mean * ws_mean
    ws_var = max(ws_var, 1e-8)

    vt_mean = vt_sum / max(1, n)
    vt_var = vt_sqs / max(1, n) - vt_mean * vt_mean
    vt_var = max(vt_var, 1e-8)

    return float(ws_mean), float(math.sqrt(ws_var)), float(vt_mean), float(math.sqrt(vt_var))

print("Computing train-only engineered stats (wind_speed, ventilation)...")
WS_MEAN, WS_STD, VT_MEAN, VT_STD = compute_engineered_stats_from_indices(TRAIN_IDX, sample_count=500, pix_per_sample=2048)
print(
    "Engineered stats:",
    "WS(mean,std)=", (round(WS_MEAN, 4), round(WS_STD, 4)),
    "VT(mean,std)=", (round(VT_MEAN, 4), round(VT_STD, 4))
)

def norm_ws(x):
    return (x - WS_MEAN) / (WS_STD + 1e-6)

def norm_vt(x):
    return (x - VT_MEAN) / (VT_STD + 1e-6)

CPM_MEAN_T = torch.tensor(MEAN[TARGET], dtype=torch.float32, device=DEVICE)
CPM_STD_T  = torch.tensor(STD[TARGET], dtype=torch.float32, device=DEVICE)

def denorm_cpm_torch(x_norm):
    return x_norm * (CPM_STD_T + 1e-6) + CPM_MEAN_T

# -----------------------
# 5) Dataset with patch training
# -----------------------
class PMDataset(Dataset):
    def __init__(self, indices, crop_size=96, train=True):
        self.indices = indices
        self.crop_size = crop_size
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        m, t0 = self.indices[idx]
        cpm25 = MONTH_DATA[m][TARGET][t0:t0+T_TOTAL].astype(np.float32)

        cov = []
        for f in COVARS:
            cov.append(MONTH_DATA[m][f][t0:t0+T_TOTAL].astype(np.float32))
        cov = np.stack(cov, axis=1)  # (26, 15, H, W)

        u10 = cov[:, MET.index("u10")]
        v10 = cov[:, MET.index("v10")]
        pblh = cov[:, MET.index("pblh")]
        wind_speed = np.sqrt(u10*u10 + v10*v10 + 1e-6).astype(np.float32)
        ventilation = (wind_speed * pblh).astype(np.float32)

        cpm25_26 = np.zeros((T_TOTAL, H, W), dtype=np.float32)
        cpm25_26[:T_CTX] = cpm25[:T_CTX]
        mask_26 = np.zeros((T_TOTAL, H, W), dtype=np.float32)
        mask_26[:T_CTX] = 1.0

        lat = np.broadcast_to(LAT_CH[None, ...], (T_TOTAL, H, W))
        lon = np.broadcast_to(LON_CH[None, ...], (T_TOTAL, H, W))

        cpm25_26_n = norm(cpm25_26, TARGET).astype(np.float32)

        cov_n = cov.copy()
        for i, f in enumerate(COVARS):
            cov_n[:, i] = norm(cov_n[:, i], f)

        wind_speed_n = norm_ws(wind_speed).astype(np.float32)
        ventilation_n = norm_vt(ventilation).astype(np.float32)

        X = np.concatenate([
            cov_n,                               # (26,15,H,W)
            wind_speed_n[:, None, ...],         # (26,1,H,W)
            ventilation_n[:, None, ...],        # (26,1,H,W)
            cpm25_26_n[:, None, ...],           # (26,1,H,W)
            mask_26[:, None, ...],              # (26,1,H,W)
            lat[:, None, ...],                  # (26,1,H,W)
            lon[:, None, ...],                  # (26,1,H,W)
        ], axis=1).astype(np.float32)           # (26,21,H,W)

        Y = norm(cpm25[T_CTX:T_TOTAL], TARGET).astype(np.float32)    # (16,H,W)
        last_ctx = norm(cpm25[T_CTX-1], TARGET).astype(np.float32)   # (H,W)
        Y_res = (Y - last_ctx[None, ...]).astype(np.float32)         # residual target

        if self.train and self.crop_size is not None:
            cs = self.crop_size
            if cs < H and cs < W:
                y0 = random.randint(0, H - cs)
                x0 = random.randint(0, W - cs)
                X = X[:, :, y0:y0+cs, x0:x0+cs]
                Y_res = Y_res[:, y0:y0+cs, x0:x0+cs]
                last_ctx = last_ctx[y0:y0+cs, x0:x0+cs]

        X = torch.from_numpy(X)
        Y_res = torch.from_numpy(Y_res)
        last_ctx = torch.from_numpy(last_ctx)
        return X, Y_res, last_ctx

# -----------------------
# 6) Model: UNet + Temporal Transformer
# -----------------------
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.GroupNorm(8 if out_ch >= 8 else 1, out_ch),
            nn.SiLU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.GroupNorm(8 if out_ch >= 8 else 1, out_ch),
            nn.SiLU(),
        )

    def forward(self, x):
        return self.net(x)

class UNetEncoder(nn.Module):
    def __init__(self, in_ch, base=32, emb=128):
        super().__init__()
        self.c1 = ConvBlock(in_ch, base)
        self.c2 = ConvBlock(base, base*2)
        self.c3 = ConvBlock(base*2, emb)
        self.p = nn.MaxPool2d(2)

    def forward(self, x):
        s1 = self.c1(x)
        x = self.p(s1)
        s2 = self.c2(x)
        x = self.p(s2)
        x = self.c3(x)
        return x, s1, s2

class UNetDecoder(nn.Module):
    def __init__(self, emb=128, base=32, out_ch=1):
        super().__init__()
        self.up1 = nn.ConvTranspose2d(emb, base*2, 2, stride=2)
        self.d1  = ConvBlock(base*2 + base*2, base*2)
        self.up2 = nn.ConvTranspose2d(base*2, base, 2, stride=2)
        self.d2  = ConvBlock(base + base, base)
        self.out = nn.Conv2d(base, out_ch, 1)

    def forward(self, x, s1, s2):
        x = self.up1(x)
        if x.shape[-2:] != s2.shape[-2:]:
            x = F.interpolate(x, size=s2.shape[-2:], mode="bilinear", align_corners=False)
        x = torch.cat([x, s2], dim=1)
        x = self.d1(x)

        x = self.up2(x)
        if x.shape[-2:] != s1.shape[-2:]:
            x = F.interpolate(x, size=s1.shape[-2:], mode="bilinear", align_corners=False)
        x = torch.cat([x, s1], dim=1)
        x = self.d2(x)
        return self.out(x)

class TemporalTransformer(nn.Module):
    def __init__(self, d_model=128, nhead=4, num_layers=2, dropout=0.1, T=26):
        super().__init__()
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model*4,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.pos = nn.Parameter(torch.randn(T, d_model) * 0.02)

    def forward(self, x):
        T = x.shape[1]
        x = x + self.pos[:T][None, :, :]
        return self.encoder(x)

class UNetTemporalModel(nn.Module):
    def __init__(self, cin, base=32, emb=128, nhead=4, nlayers=2):
        super().__init__()
        self.enc = UNetEncoder(cin, base=base, emb=emb)
        self.tr  = TemporalTransformer(d_model=emb, nhead=nhead, num_layers=nlayers, T=T_TOTAL)
        self.dec = UNetDecoder(emb=emb, base=base, out_ch=1)

    def forward(self, X):
        B, T, Cin, Hx, Wx = X.shape
        feats = []
        s1_list = []
        s2_list = []

        for t in range(T):
            f, s1, s2 = self.enc(X[:, t])
            feats.append(f)
            s1_list.append(s1)
            s2_list.append(s2)

        Fenc = torch.stack(feats, dim=1)  # (B,T,emb,h4,w4)
        h4, w4 = Fenc.shape[-2], Fenc.shape[-1]

        tokens = Fenc.permute(0, 3, 4, 1, 2).contiguous().view(B*h4*w4, T, -1)
        tokens = self.tr(tokens)
        Ftr = tokens.view(B, h4, w4, T, -1).permute(0, 3, 4, 1, 2).contiguous()

        outs = []
        for k in range(T_CTX, T_TOTAL):
            xk = Ftr[:, k]
            s1k = s1_list[k]
            s2k = s2_list[k]
            yk = self.dec(xk, s1k, s2k)[:, 0]
            outs.append(yk)

        Y = torch.stack(outs, dim=1)  # (B,16,H,W)
        return Y

# -----------------------
# 7) Hyperparameter tuning setup
# -----------------------
CIN = 21

HYPERPARAM_TRIALS = [
    {"name": "trial1", "base": 32, "emb": 128, "nlayers": 2, "nhead": 4, "lr": 2e-4, "wd": 1e-4, "crop": 96, "bs": 4},

]

NUM_WORKERS = 2
INF_BS = 2
EPOCHS = 25
PATIENCE = 4
VAL_MAX_BATCHES = 120
FULL_RETRAIN_EPOCHS = 12

def build_loaders(indices_train, indices_val=None, crop_size=96, batch_size=4):
    train_ds = PMDataset(indices_train, crop_size=crop_size, train=True)
    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        drop_last=True,
        persistent_workers=(NUM_WORKERS > 0),
        prefetch_factor=2 if NUM_WORKERS > 0 else None
    )

    val_loader = None
    if indices_val is not None:
        val_ds = PMDataset(indices_val, crop_size=None, train=False)
        val_loader = DataLoader(
            val_ds,
            batch_size=1,
            shuffle=False,
            num_workers=NUM_WORKERS,
            pin_memory=True,
            persistent_workers=(NUM_WORKERS > 0),
            prefetch_factor=2 if NUM_WORKERS > 0 else None
        )
    return train_loader, val_loader

def evaluate(model, val_loader):
    model.eval()
    rmses = []
    with torch.inference_mode():
        for i, (X, Y_res, last_ctx) in enumerate(val_loader):
            if i >= VAL_MAX_BATCHES:
                break

            X = X.to(DEVICE, non_blocking=True)
            Y_res = Y_res.to(DEVICE, non_blocking=True)
            last_ctx = last_ctx.to(DEVICE, non_blocking=True)

            with torch.amp.autocast(device_type="cuda", enabled=(DEVICE == "cuda")):
                pred_res = model(X)
                pred_abs_norm = pred_res + last_ctx[:, None, :, :]
                true_abs_norm = Y_res + last_ctx[:, None, :, :]

                pred_phys = denorm_cpm_torch(pred_abs_norm)
                true_phys = denorm_cpm_torch(true_abs_norm)

                rmse = torch.sqrt(torch.mean((pred_phys - true_phys) ** 2) + 1e-8)

            rmses.append(rmse.item())

    return float(np.mean(rmses)) if rmses else float("inf")

def train_one_trial(tcfg):
    train_loader, val_loader = build_loaders(TRAIN_IDX, VAL_IDX, crop_size=tcfg["crop"], batch_size=tcfg["bs"])

    model = UNetTemporalModel(
        cin=CIN,
        base=tcfg["base"],
        emb=tcfg["emb"],
        nhead=tcfg["nhead"],
        nlayers=tcfg["nlayers"]
    ).to(DEVICE)

    print("Model params:", sum(p.numel() for p in model.parameters()) / 1e6, "M")

    opt = torch.optim.AdamW(model.parameters(), lr=tcfg["lr"], weight_decay=tcfg["wd"])
    scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE == "cuda"))
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

    trial_best = float("inf")
    bad = 0
    best_epoch = 1
    trial_ckpt_path = f"/kaggle/working/best_model_{tcfg['name']}.pt"

    for ep in range(1, EPOCHS + 1):
        t0 = time.time()
        model.train()
        running = 0.0
        n = 0

        for X, Y_res, last_ctx in train_loader:
            X = X.to(DEVICE, non_blocking=True)
            Y_res = Y_res.to(DEVICE, non_blocking=True)
            last_ctx = last_ctx.to(DEVICE, non_blocking=True)

            opt.zero_grad(set_to_none=True)

            with torch.amp.autocast(device_type="cuda", enabled=(DEVICE == "cuda")):
                pred_res = model(X)
                pred_abs_norm = pred_res + last_ctx[:, None, :, :]
                true_abs_norm = Y_res + last_ctx[:, None, :, :]

                pred_phys = denorm_cpm_torch(pred_abs_norm)
                true_phys = denorm_cpm_torch(true_abs_norm)

                loss = torch.sqrt(torch.mean((pred_phys - true_phys) ** 2) + 1e-8)

            scaler.scale(loss).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()

            running += loss.item()
            n += 1

        scheduler.step()
        train_loss = running / max(1, n)
        val_metric = evaluate(model, val_loader)

        dt = time.time() - t0
        print(f"Epoch {ep:02d} | train_domain_rmse={train_loss:.5f} | val_domain_rmse={val_metric:.5f} | time={dt:.1f}s")

        if val_metric < trial_best - 1e-6:
            trial_best = val_metric
            best_epoch = ep
            bad = 0
            torch.save(
                {
                    "model": model.state_dict(),
                    "cfg": tcfg,
                    "best_epoch": best_epoch,
                    "MEAN": MEAN, "STD": STD,
                    "WS_MEAN": WS_MEAN, "WS_STD": WS_STD,
                    "VT_MEAN": VT_MEAN, "VT_STD": VT_STD
                },
                trial_ckpt_path
            )
            print("  saved", trial_ckpt_path)
        else:
            bad += 1
            if bad >= PATIENCE:
                print("Early stopping for this trial.")
                break

    del model, opt, scaler, scheduler, train_loader, val_loader
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    return trial_best, trial_ckpt_path, best_epoch

best_trial_metric = float("inf")
best_trial_path = None
best_trial_cfg = None
best_trial_epoch = None

print("\n====================")
print("Hyperparameter tuning starts")
print("====================\n")

for tcfg in HYPERPARAM_TRIALS:
    print("\n--------------------")
    print("Running:", tcfg)
    print("--------------------")

    trial_best, trial_ckpt_path, best_epoch = train_one_trial(tcfg)
    print("Trial best val_domain_rmse:", trial_best)
    print("Trial best epoch:", best_epoch)

    if trial_best < best_trial_metric:
        best_trial_metric = trial_best
        best_trial_path = trial_ckpt_path
        best_trial_cfg = tcfg
        best_trial_epoch = best_epoch

print("\n====================")
print("Hyperparameter tuning finished")
print("====================")
print("Best trial metric:", best_trial_metric)
print("Best trial checkpoint:", best_trial_path)
print("Best trial config:", best_trial_cfg)
print("Best trial epoch:", best_trial_epoch)

# -----------------------
# 8) Rebuild best config and train again on all months
# -----------------------
print("\n====================")
print("Full-data retraining starts")
print("====================")

best_cfg = best_trial_cfg
full_epochs = min(best_trial_epoch, FULL_RETRAIN_EPOCHS)

print("Retraining best config on ALL months")
print("Using epochs:", full_epochs)

full_train_loader, _ = build_loaders(FULL_IDX, None, crop_size=best_cfg["crop"], batch_size=best_cfg["bs"])

final_model = UNetTemporalModel(
    cin=CIN,
    base=best_cfg["base"],
    emb=best_cfg["emb"],
    nhead=best_cfg["nhead"],
    nlayers=best_cfg["nlayers"]
).to(DEVICE)

print("Final model params:", sum(p.numel() for p in final_model.parameters()) / 1e6, "M")

final_opt = torch.optim.AdamW(final_model.parameters(), lr=best_cfg["lr"], weight_decay=best_cfg["wd"])
final_scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE == "cuda"))
final_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(final_opt, T_max=full_epochs)

for ep in range(1, full_epochs + 1):
    t0 = time.time()
    final_model.train()
    running = 0.0
    n = 0

    for X, Y_res, last_ctx in full_train_loader:
        X = X.to(DEVICE, non_blocking=True)
        Y_res = Y_res.to(DEVICE, non_blocking=True)
        last_ctx = last_ctx.to(DEVICE, non_blocking=True)

        final_opt.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type="cuda", enabled=(DEVICE == "cuda")):
            pred_res = final_model(X)
            pred_abs_norm = pred_res + last_ctx[:, None, :, :]
            true_abs_norm = Y_res + last_ctx[:, None, :, :]

            pred_phys = denorm_cpm_torch(pred_abs_norm)
            true_phys = denorm_cpm_torch(true_abs_norm)

            loss = torch.sqrt(torch.mean((pred_phys - true_phys) ** 2) + 1e-8)

        final_scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(final_model.parameters(), 1.0)
        final_scaler.step(final_opt)
        final_scaler.update()

        running += loss.item()
        n += 1

    final_scheduler.step()
    dt = time.time() - t0
    train_loss = running / max(1, n)
    print(f"[FULL] Epoch {ep:02d}/{full_epochs} | train_domain_rmse={train_loss:.5f} | time={dt:.1f}s")

final_model_path = "/kaggle/working/final_model_full_retrain.pt"
torch.save(
    {
        "model": final_model.state_dict(),
        "cfg": best_cfg,
        "epochs": full_epochs,
        "MEAN": MEAN, "STD": STD,
        "WS_MEAN": WS_MEAN, "WS_STD": WS_STD,
        "VT_MEAN": VT_MEAN, "VT_STD": VT_STD
    },
    final_model_path
)
print("Saved final retrained model to:", final_model_path)

del full_train_loader, final_opt, final_scaler, final_scheduler
if DEVICE == "cuda":
    torch.cuda.empty_cache()

# -----------------------
# 9) Final inference with fully retrained model
# -----------------------
print("\n====================")
print("Final inference starts")
print("====================")

def load_test_feature(name):
    path = os.path.join(TEST_PATH, f"{name}.npy")
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing test feature {path}")
    return np.load(path, mmap_mode="r")

test_cpm25 = load_test_feature("cpm25")
N_TEST = test_cpm25.shape[0]
print("Test cpm25 shape:", test_cpm25.shape)

test_cov = {}
for f in COVARS:
    test_cov[f] = load_test_feature(f)
print("Loaded test covars:", len(test_cov))

preds_all = []

lat_26 = np.broadcast_to(LAT_CH[None, None, ...], (1, T_TOTAL, H, W)).astype(np.float32)
lon_26 = np.broadcast_to(LON_CH[None, None, ...], (1, T_TOTAL, H, W)).astype(np.float32)

final_model.eval()
with torch.inference_mode():
    for s in range(0, N_TEST, INF_BS):
        e = min(N_TEST, s + INF_BS)
        bsz = e - s

        cpm25_26 = np.zeros((bsz, T_TOTAL, H, W), dtype=np.float32)
        cpm25_26[:, :T_CTX] = np.array(test_cpm25[s:e], dtype=np.float32)

        mask_26 = np.zeros((bsz, T_TOTAL, H, W), dtype=np.float32)
        mask_26[:, :T_CTX] = 1.0

        cov_list = []
        for f in COVARS:
            cov_list.append(np.array(test_cov[f][s:e], dtype=np.float32))
        cov = np.stack(cov_list, axis=2)  # (B,26,15,H,W)

        u10 = cov[:, :, MET.index("u10")]
        v10 = cov[:, :, MET.index("v10")]
        pblh = cov[:, :, MET.index("pblh")]
        wind_speed = np.sqrt(u10*u10 + v10*v10 + 1e-6).astype(np.float32)
        ventilation = (wind_speed * pblh).astype(np.float32)

        cpm25_26_n = norm(cpm25_26, TARGET).astype(np.float32)

        cov_n = cov.copy()
        for i, f in enumerate(COVARS):
            cov_n[:, :, i] = norm(cov_n[:, :, i], f)

        ws_n = norm_ws(wind_speed).astype(np.float32)
        vt_n = norm_vt(ventilation).astype(np.float32)

        lat_b = np.repeat(lat_26, bsz, axis=0)
        lon_b = np.repeat(lon_26, bsz, axis=0)

        X = np.concatenate([
            cov_n,
            ws_n[:, :, None, ...],
            vt_n[:, :, None, ...],
            cpm25_26_n[:, :, None, ...],
            mask_26[:, :, None, ...],
            lat_b[:, :, None, ...],
            lon_b[:, :, None, ...],
        ], axis=2).astype(np.float32)  # (B,26,21,H,W)

        X_t = torch.from_numpy(X).to(DEVICE, non_blocking=True)

        with torch.amp.autocast(device_type="cuda", enabled=(DEVICE == "cuda")):
            pred_res = final_model(X_t)

        last_ctx = norm(np.array(test_cpm25[s:e, -1], dtype=np.float32), TARGET)
        last_ctx_t = torch.from_numpy(last_ctx).to(DEVICE, non_blocking=True)

        pred_norm = pred_res + last_ctx_t[:, None, :, :]
        pred = pred_norm.detach().cpu().numpy()
        pred = denorm(pred, TARGET).astype(np.float32)

        preds_all.append(pred)

preds = np.concatenate(preds_all, axis=0)  # (N,16,H,W)
print("Preds (N,16,H,W):", preds.shape)

preds_out = np.transpose(preds, (0, 2, 3, 1))  # (N,H,W,16)
assert preds_out.shape == (N_TEST, H, W, T_FCST), preds_out.shape

out_path = "/kaggle/working/preds.npy"
np.save(out_path, preds_out)

print("Saved:", out_path, "shape:", preds_out.shape)
print("File size MB:", os.path.getsize(out_path) / (1024**2))

/usr/local/lib/python3.12/dist-packages/torch/__init__.py:1617: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  _C._set_float32_matmul_precision(precision)


DEVICE: cuda
RAW_PATH: /kaggle/input/competitions/aisehack-theme-2/raw
TEST_PATH: /kaggle/input/competitions/aisehack-theme-2/test_in
Month folders in raw: ['APRIL_16', 'DEC_16', 'JULY_16', 'OCT_16']
Using months: ['APRIL_16', 'JULY_16', 'OCT_16', 'DEC_16']
lat_long.npy shape: (140, 124, 2)
Grid: 140 124
APRIL_16 cpm25 (715, 140, 124)
JULY_16 cpm25 (739, 140, 124)
OCT_16 cpm25 (739, 140, 124)
DEC_16 cpm25 (739, 140, 124)
Train windows: 2241 Val windows: 591 Full windows: 2832
Computing train-only normalization stats (fast sampling)...
Example stats: {'cpm25': (33.446, 52.286), 'q2': (0.012, 0.007), 't2': (291.677, 13.831), 'u10': (1.694, 3.571), 'v10': (0.158, 3.06)}
Computing train-only engineered stats (wind_speed, ventilation)...
Engineered stats: WS(mean,std)= (4.2575, 2.6246) VT(mean,std)= (3475.6419, 4136.0399)

Hyperparameter tuning starts


--------------------
Running: {'name': 'trial1', 'base': 32, 'emb': 128, 'nlayers': 2, 'nhead': 4, 'lr': 0.0002, 'wd': 0.0001, 'crop': 96, 

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Epoch 01 | train_domain_rmse=23.30315 | val_domain_rmse=12.30772 | time=194.5s
  saved /kaggle/working/best_model_trial1.pt
Epoch 02 | train_domain_rmse=18.63452 | val_domain_rmse=11.06775 | time=174.9s
  saved /kaggle/working/best_model_trial1.pt
Epoch 03 | train_domain_rmse=16.79539 | val_domain_rmse=10.33388 | time=174.6s
  saved /kaggle/working/best_model_trial1.pt
Epoch 04 | train_domain_rmse=15.57139 | val_domain_rmse=9.91611 | time=175.3s
  saved /kaggle/working/best_model_trial1.pt
Epoch 05 | train_domain_rmse=14.22809 | val_domain_rmse=9.68946 | time=174.7s
  saved /kaggle/working/best_model_trial1.pt
Epoch 06 | train_domain_rmse=13.78008 | val_domain_rmse=9.58163 | time=175.0s
  saved /kaggle/working/best_model_trial1.pt
Epoch 07 | train_domain_rmse=13.19545 | val_domain_rmse=9.52020 | time=174.8s
  saved /kaggle/working/best_model_trial1.pt
Epoch 08 | train_domain_rmse=12.99604 | val_domain_rmse=9.70706 | time=175.3s
Epoch 09 | train_domain_rmse=12.37489 | val_domain_rmse=9.

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


[FULL] Epoch 01/12 | train_domain_rmse=23.76739 | time=209.2s
[FULL] Epoch 02/12 | train_domain_rmse=18.96949 | time=207.6s
[FULL] Epoch 03/12 | train_domain_rmse=17.38530 | time=207.9s
[FULL] Epoch 04/12 | train_domain_rmse=15.90820 | time=208.1s
[FULL] Epoch 05/12 | train_domain_rmse=14.46649 | time=208.3s
[FULL] Epoch 06/12 | train_domain_rmse=13.45880 | time=207.8s
[FULL] Epoch 07/12 | train_domain_rmse=12.79881 | time=208.7s
[FULL] Epoch 08/12 | train_domain_rmse=12.33459 | time=208.2s
[FULL] Epoch 09/12 | train_domain_rmse=12.07742 | time=207.9s
[FULL] Epoch 10/12 | train_domain_rmse=11.84053 | time=208.1s
[FULL] Epoch 11/12 | train_domain_rmse=11.53623 | time=208.0s
[FULL] Epoch 12/12 | train_domain_rmse=11.53096 | time=208.0s
Saved final retrained model to: /kaggle/working/final_model_full_retrain.pt

Final inference starts
Test cpm25 shape: (996, 10, 140, 124)
Loaded test covars: 15
Preds (N,16,H,W): (996, 16, 140, 124)
Saved: /kaggle/working/preds.npy shape: (996, 140, 124, 1